In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output

import warnings
warnings.filterwarnings("ignore")
sns.set(style='whitegrid')



In [3]:
uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])
df.columns = df.columns.str.strip()


Saving global-data-on-sustainable-energy (1).csv to global-data-on-sustainable-energy (1).csv


In [4]:
def forecast_country(country):
    df_country = df[df['Entity'] == country].copy()

    try:
        df_country = df_country[[
            'Year',
            'Electricity from renewables (TWh)',
            'Electricity from fossil fuels (TWh)',
            'Electricity from nuclear (TWh)',
            'gdp_per_capita',
            'Access to electricity (% of population)',
            'Renewable energy share in the total final energy consumption (%)'
        ]].dropna()
    except:
        print(f"⚠️ Skipping {country}: missing required data.")
        return None

    # addded features
    df_country['Renewables_prev_year'] = df_country['Electricity from renewables (TWh)'].shift(1)
    df_country['Renewables_growth_rate'] = df_country['Electricity from renewables (TWh)'].pct_change() * 100
    df_country.dropna(inplace=True)

    # prophet-ready format
    df_prophet = df_country[['Year', 'Electricity from renewables (TWh)']].rename(
        columns={'Year': 'ds', 'Electricity from renewables (TWh)': 'y'})
    df_prophet['ds'] = pd.to_datetime(df_prophet['ds'], format='%Y')

    # training Prophet
    model = Prophet()
    model.fit(df_prophet)

    # Forecasting
    future = model.make_future_dataframe(periods=10, freq='Y')
    forecast = model.predict(future)

    # Plottingg
    fig = model.plot(forecast)
    plt.title(f"{country}: Renewable Electricity Forecast")
    plt.xlabel("Year")
    plt.ylabel("TWh")
    plt.show()

    return forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10).assign(Country=country)


In [5]:
# Dropdown + Button
countries = sorted(df['Entity'].dropna().unique())
country_dropdown = widgets.Dropdown(options=countries, description='Select Country:')
run_button = widgets.Button(description='Forecast')

output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output(wait=True)
        selected_country = country_dropdown.value
        print(f"Forecasting for: {selected_country}")
        forecast_df = forecast_country(selected_country)
        if forecast_df is not None:
            display(forecast_df)

run_button.on_click(on_button_clicked)

# Display UI
display(widgets.VBox([country_dropdown, run_button, output]))


In [ ]:
def forecast_multiple_features(country, columns_to_forecast):
    df_country = df[df['Entity'] == country].copy()
    df_country = df_country[['Year'] + columns_to_forecast].dropna()
    df_country['Year'] = pd.to_datetime(df_country['Year'], format='%Y')

    for col in columns_to_forecast:
        df_feat = df_country[['Year', col]].rename(columns={'Year': 'ds', col: 'y'})
        model = Prophet()
        model.fit(df_feat)

        future = model.make_future_dataframe(periods=10, freq='Y')
        forecast = model.predict(future)

        # Plot
        model.plot(forecast)
        plt.title(f"{country}: Forecast for {col}")
        plt.xlabel("Year")
        plt.ylabel(col)
        plt.show()


In [ ]:
features_to_forecast = [
    'Access to electricity (% of population)',
    'Access to clean fuels for cooking',
    'Renewable-electricity-generating-capacity-per-capita',
    'Financial flows to developing countries (US $)',
    'Renewable energy share in the total final energy consumption (%)',
    'Electricity from fossil fuels (TWh)',
    'Electricity from nuclear (TWh)',
    'Electricity from renewables (TWh)',
    'Low-carbon electricity (% electricity)',
    'Primary energy consumption per capita (kWh/person)',
    'Energy intensity level of primary energy (MJ/$2017 PPP GDP)',
    'Value_co2_emissions_kt_by_country',
    'Renewables (% equivalent primary energy)',
    'gdp_growth',
    'gdp_per_capita'
]


In [ ]:
countries = sorted(df['Entity'].dropna().unique())
dropdown = widgets.Dropdown(options=countries, description='Country:')
button = widgets.Button(description='Run Forecast')
output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output(wait=True)
        selected_country = dropdown.value
        print(f"Forecasting multiple indicators for {selected_country}")
        forecast_multiple_features(selected_country, features_to_forecast)

button.on_click(on_button_clicked)

display(widgets.VBox([dropdown, button, output]))
